### Fase 1: Aquisição, Limpeza e Enriquecimento de Dados

In [1]:
# Bibliotecas
import pandas as pd
import re

In [2]:
# Carregando as bases de dados

df_reviews = pd.read_csv('olist_order_reviews_dataset.csv')
df_orders = pd.read_csv('olist_orders_dataset.csv')
df_items = pd.read_csv('olist_order_items_dataset.csv')
df_products = pd.read_csv('olist_products_dataset.csv')

print(f"Total de avaliações originais: {len(df_reviews)}")

Total de avaliações originais: 99224


### Filtragem e Cruzamento(Joins) 

* Necessários para o desafio bônus

In [3]:
# 1. Manter apenas avaliações que possuam comentário em texto
df_base = df_reviews.dropna(subset=['review_comment_message']).copy()
print(f"Avaliações com texto: {len(df_base)}")

# 2. Trazer a data do pedido (para análises temporais no RAG)
df_base = df_base.merge(df_orders[['order_id', 'order_purchase_timestamp']], on='order_id', how='left')

# 3. Trazer o ID do produto
# Um pedido pode ter vários itens iguais, usamos drop_duplicates para não duplicar a avaliação
df_items_unique = df_items[['order_id', 'product_id']].drop_duplicates(subset=['order_id'])
df_base = df_base.merge(df_items_unique, on='order_id', how='left')

# 4. Trazer a categoria do produto (Metadata bônus do desafio)
df_base = df_base.merge(df_products[['product_id', 'product_category_name']], on='product_id', how='left')

# Preencher categorias vazias com 'desconhecido'
df_base['product_category_name'] = df_base['product_category_name'].fillna('desconhecido')

df_base.head(3)

Avaliações com texto: 40977


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,order_purchase_timestamp,product_id,product_category_name
0,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06,2017-04-09 17:41:13,52c80cedd4e90108bf4fa6a206ef6b03,ferramentas_jardim
1,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53,2018-02-10 10:59:03,3880d25d502b15b1de6fddc42ad1d67a,esporte_lazer
2,8670d52e15e00043ae7de4c01cc2fe06,b9bf720beb4ab3728760088589c62129,4,recomendo,aparelho eficiente. no site a marca do aparelh...,2018-05-22 00:00:00,2018-05-23 16:45:47,2018-05-14 10:29:02,61a4100ccd6d9c4c808a1fd954ddb8ad,eletroportateis


### Limpeza e Preparação dos Textos

In [4]:
def limpar_texto(texto):
    if pd.isna(texto):
        return ""
    texto = str(texto)
    # Remove quebras de linha
    texto = re.sub(r'[\r\n]+', ' ', texto)
    # Remove espaços duplos
    texto = re.sub(r'\s+', ' ', texto)
    return texto.strip()

# Aplicar limpeza no título e na mensagem
df_base['review_comment_title'] = df_base['review_comment_title'].apply(limpar_texto)
df_base['review_comment_message'] = df_base['review_comment_message'].apply(limpar_texto)

# Criar a coluna final de texto (Contexto) que será vetorizada
# Se tiver título, junta com a mensagem. Se não, usa só a mensagem.
df_base['texto_completo'] = df_base.apply(
    lambda row: f"{row['review_comment_title']} - {row['review_comment_message']}" if row['review_comment_title'] != "" else row['review_comment_message'],
    axis=1
)

# Converter a data para formato datetime e extrair apenas ano/mês (facilita o LLM)
df_base['order_purchase_timestamp'] = pd.to_datetime(df_base['order_purchase_timestamp'])
df_base['ano_mes'] = df_base['order_purchase_timestamp'].dt.strftime('%Y-%m')

df_base[['review_score', 'product_category_name', 'texto_completo']].head()

,review_score,product_category_name,texto_completo
0,5,ferramentas_jardim,Recebi bem antes do prazo estipulado.
1,5,esporte_lazer,Parabéns lojas lannister adorei comprar pela I...
2,4,eletroportateis,recomendo - aparelho eficiente. no site a marc...
3,4,beleza_saude,"Mas um pouco ,travando...pelo valor ta Boa."
4,5,informatica_acessorios,"Super recomendo - Vendedor confiável, produto ..."


### Seleção Final e Exportação

In [5]:
# Selecionar apenas as colunas relevantes para o banco vetorial
colunas_finais = [
    'review_id', 
    'order_id', 
    'review_score', 
    'product_category_name', 
    'ano_mes', 
    'texto_completo'
]

df_voce_cliente = df_base[colunas_finais].copy()

# Remover textos muito curtos (ex: "ok", "bom") que não agregam valor semântico ao RAG
# Vamos definir um limite mínimo de 15 caracteres
df_voce_cliente = df_voce_cliente[df_voce_cliente['texto_completo'].str.len() > 15]

print(f"Base final limpa e pronta para embeddings: {len(df_voce_cliente)} avaliações")

# Exportar para não precisar processar tudo de novo na Fase 2
df_voce_cliente.to_csv('base_conhecimento_limpa.csv', index=False)

Base final limpa e pronta para embeddings: 36513 avaliações


### Fase 2: Criação dos Embeddings e do Banco Vetorial.

In [ ]:
!pip install langchain langchain-community langchain-huggingface sentence-transformers chromadb tqdm

In [6]:
import pandas as pd
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from tqdm import tqdm

C:\Users\gugag\AppData\Local\Temp\ipykernel_18204\3444449747.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


In [7]:
# 1. Carregar a base limpa gerada na Fase 1
df_voce_cliente = pd.read_csv('base_conhecimento_limpa.csv')

# DICA DE TESTE: Se quiser validar se tudo funciona antes de esperar os 20 minutos, 
# descomente a linha abaixo para testar com apenas 1000 linhas. Depois que der certo, 
# comente novamente e rode com a base inteira.
# df_voce_cliente = df_voce_cliente.sample(1000, random_state=42)

# 2. Transformar o DataFrame em objetos Document do LangChain
documentos = []
for _, row in tqdm(df_voce_cliente.iterrows(), total=len(df_voce_cliente), desc="Preparando documentos"):
    doc = Document(
        page_content=str(row['texto_completo']), # O texto que será buscado
        metadata={
            "review_id": str(row['review_id']),
            "order_id": str(row['order_id']),
            "review_score": int(row['review_score']),
            "product_category_name": str(row['product_category_name']),
            "ano_mes": str(row['ano_mes'])
        }
    )
    documentos.append(doc)

# 3. Inicializar o modelo de Embeddings
print("\nBaixando/Carregando o modelo de embeddings...")
embeddings_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

# 4. Criar e salvar o Banco Vetorial
print("Iniciando a vetorização e criação do ChromaDB. Pode ir pegar um café, isso vai levar alguns minutos...")
vectordb = Chroma.from_documents(
    documents=documentos, 
    embedding=embeddings_model, 
    persist_directory="./chroma_db_olist" # Salva localmente nesta pasta
)

print("\nSucesso! Banco vetorial criado e salvo na pasta 'chroma_db_olist'.")

Preparando documentos: 100%|██████████████████████████████████████████████████| 36513/36513 [00:01<00:00, 18904.43it/s]



Baixando/Carregando o modelo de embeddings...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Iniciando a vetorização e criação do ChromaDB. Pode ir pegar um café, isso vai levar alguns minutos...

Sucesso! Banco vetorial criado e salvo na pasta 'chroma_db_olist'.


In [8]:
# Testando a recuperação de dados (Retrieval)
retriever = vectordb.as_retriever(search_kwargs={"k": 3}) # Traz os 3 resultados mais relevantes

pergunta_teste = "O que os clientes estão falando sobre atraso na entrega?"
resultados = retriever.invoke(pergunta_teste)

print(f"Pergunta: {pergunta_teste}\n")
for i, res in enumerate(resultados, 1):
    print(f"--- Evidência {i} ---")
    print(f"Texto: {res.page_content}")
    print(f"Nota: {res.metadata['review_score']} | Categoria: {res.metadata['product_category_name']}")
    print("-" * 20)

Pergunta: O que os clientes estão falando sobre atraso na entrega?

--- Evidência 1 ---
Texto: sou cliente faz anos, é a 1ª vez que acontece atraso de entrega de mercadorias
Nota: 4 | Categoria: relogios_presentes
--------------------
--- Evidência 2 ---
Texto: sou cliente faz anos, é a 1ª vez que acontece atraso de entrega de mercadorias
Nota: 4 | Categoria: relogios_presentes
--------------------
--- Evidência 3 ---
Texto: demora na entrega
Nota: 1 | Categoria: beleza_saude
--------------------


### Fase 3: Pipeline RAG (Geração de Resposta).

In [ ]:
!pip install langchain.prompts

In [9]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [10]:
# Carrega as variáveis do arquivo .env
load_dotenv()

# Verifica se a chave foi carregada com sucesso
if "GOOGLE_API_KEY" in os.environ:
    print("Chave de API carregada com sucesso!")
else:
    print("ERRO: A variável GOOGLE_API_KEY não foi encontrada. Verifique seu arquivo .env.")

Chave de API carregada com sucesso!


In [ ]:
!pip install google-generativeai

In [11]:
# Modelos disponíveis:
import google.generativeai as genai
import os
from dotenv import load_dotenv

# Garante que as variáveis de ambiente estão carregadas
load_dotenv()

# Configura a biblioteca nativa do Google com a sua chave
genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

print("Modelos disponíveis para geração de texto na sua chave:\n")

# Lista todos os modelos disponíveis e filtra os que geram conteúdo
for m in genai.list_models():
    if 'generateContent' in m.supported_generation_methods:
        print(m.name)

C:\Users\gugag\AppData\Local\Temp\ipykernel_18204\2814403209.py:2: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


Modelos disponíveis para geração de texto na sua chave:

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-nano-banana-2.1
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-4-argon
models

In [12]:
# 1. Instanciar o LLM (Gemini 1.5 Flash para respostas rápidas e baratas)
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash", temperature=0.1)

# 2. Criar o Prompt System com as amarras de segurança exigidas no desafio
template_voc = """
Você é um analista de Voice of Customer (VoC) de um e-commerce.
Seu objetivo é responder à pergunta do gestor utilizando estritamente o contexto de avaliações fornecido.

Regras de negócio:
1. Responda baseando-se APENAS no contexto abaixo.
2. Se o contexto não contiver informações suficientes ou a pergunta fugir do escopo, responda EXATAMENTE: "Não possuo evidências suficientes na base de avaliações para responder a esta pergunta."
3. Sempre cite as evidências: mencione trechos dos comentários e faça referência à Categoria do produto e à Nota atribuída para validar seu ponto.

Contexto (Avaliações recuperadas):
{context}

Pergunta do gestor: {question}

Resposta analítica:
"""
prompt = PromptTemplate.from_template(template_voc)

# 3. Função para injetar os metadados no contexto lido pelo LLM
def formatar_docs_com_metadados(docs):
    contextos_formatados = []
    for d in docs:
        nota = d.metadata.get('review_score', 'N/A')
        categoria = d.metadata.get('product_category_name', 'N/A')
        texto = d.page_content
        contextos_formatados.append(f"[Nota: {nota} | Categoria: {categoria}] Comentário: {texto}")
    return "\n\n".join(contextos_formatados)

# 4. Construir a chain (LCEL)
rag_chain = (
    {"context": retriever | formatar_docs_com_metadados, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 5. Testando o Pipeline Final
pergunta_valida = "Quais problemas os clientes relatam sobre a entrega?"
print(f"Pergunta: {pergunta_valida}\n")
print(rag_chain.invoke(pergunta_valida))

print("\n" + "-" * 50 + "\n")

# Testando a trava de segurança (Requisito 4 do desafio)
pergunta_fora_escopo = "Qual a receita para fazer um bolo de chocolate?"
print(f"Pergunta: {pergunta_fora_escopo}\n")
print(rag_chain.invoke(pergunta_fora_escopo))

Pergunta: Quais problemas os clientes relatam sobre a entrega?

Com base no histórico de avaliações dos clientes, os principais problemas relatados sobre a entrega são:

1. **Entregas parciais/incompletas e divergência de status:**
   * **Evidência:** Na categoria **cool_stuff** (Nota: 1), o cliente relata a ocorrência recorrente de *"divergências e problemas na entrega"*, exemplificando que o sistema indica o pedido como concluído, mas o envio é feito de forma incompleta: *"consta como produto entregue, porém foi entregue apenas 1 produto, aguardo solução!"*.

2. **Desencontros nas entregas:**
   * **Evidência:** Na categoria **cama_mesa_banho** (Nota: 1), o cliente aponta falta de alinhamento ou falhas logísticas ao mencionar brevemente: *"muitos desencontros nas entregas de produtos"*.

--------------------------------------------------

Pergunta: Qual a receita para fazer um bolo de chocolate?

Não possuo evidências suficientes na base de avaliações para responder a esta pergunta.


### O Código da Aplicação(Agente no AR)

In [15]:
## %%writefile app.py
import streamlit as st
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

In [17]:
# 1. Configuração da Página do Streamlit
st.set_page_config(page_title="VoC Intelligence", page_icon="📊", layout="centered")
st.title("📊 Voice of Customer Intelligence")
st.markdown("Exploração inteligente de avaliações do e-commerce utilizando Retrieval-Augmented Generation (RAG).")

# 2. Carregar variáveis de ambiente
load_dotenv()
if "GOOGLE_API_KEY" not in os.environ:
    st.error("Erro: GOOGLE_API_KEY não encontrada. Verifique o arquivo .env na mesma pasta do app.py.")
    st.stop()

# 3. Funções Lazy (Carregamento Sob Demanda)
def carregar_banco_vetorial():
    # Importações feitas apenas quando necessárias (resolve o problema da tela branca)
    from langchain_huggingface import HuggingFaceEmbeddings
    from langchain_community.vectorstores import Chroma
    
    embeddings_model = HuggingFaceEmbeddings(
        model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    )
    # Garante que ele procure a pasta local criada no passo anterior
    vectordb = Chroma(persist_directory="./chroma_db_olist", embedding_function=embeddings_model)
    return vectordb.as_retriever(search_kwargs={"k": 5})

def formatar_docs_com_metadados(docs):
    contextos = []
    for d in docs:
        nota = d.metadata.get('review_score', 'N/A')
        categoria = d.metadata.get('product_category_name', 'N/A')
        texto = d.page_content
        contextos.append(f"[Nota: {nota} | Categoria: {categoria}] Comentário: {texto}")
    return "\n\n".join(contextos)

def montar_pipeline(retriever):
    llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash", temperature=0.1)
    
    template_voc = """
    Você é um analista de Voice of Customer (VoC) de um e-commerce.
    Seu objetivo é responder à pergunta do gestor utilizando estritamente o contexto de avaliações fornecido.

    Regras de negócio:
    1. Responda baseando-se APENAS no contexto abaixo.
    2. Se o contexto não contiver informações suficientes ou a pergunta fugir do escopo, responda EXATAMENTE: "Não possuo evidências suficientes na base de avaliações para responder a esta pergunta."
    3. Sempre cite as evidências: mencione trechos dos comentários e faça referência à Categoria do produto e à Nota atribuída para validar seu ponto.
    4. Estruture a resposta em tópicos para facilitar a leitura.

    Contexto (Avaliações recuperadas):
    {context}

    Pergunta do gestor: {question}

    Resposta analítica:
    """
    prompt = PromptTemplate.from_template(template_voc)
    
    rag_chain = (
        {"context": retriever | formatar_docs_com_metadados, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )
    return rag_chain

# 4. Interface de Interação
pergunta_usuario = st.text_input(
    "O que você deseja saber sobre a opinião dos clientes?", 
    placeholder="Ex: Quais são as principais reclamações sobre a entrega?"
)

if st.button("Analisar Sentimento e Avaliações"):
    if pergunta_usuario:
        with st.spinner("Conectando ao banco de avaliações e gerando análise... (isso pode levar alguns segundos)"):
            try:
                # Carrega tudo apenas quando o botão é clicado
                retriever = carregar_banco_vetorial()
                rag_chain = montar_pipeline(retriever)
                
                # Executa a geração
                resposta = rag_chain.invoke(pergunta_usuario)
                
                st.markdown("### 📋 Análise Gerencial")
                st.write(resposta)
                
                with st.expander("🔍 Ver Evidências Recuperadas (Contexto Bruto)"):
                    docs_recuperados = retriever.invoke(pergunta_usuario)
                    for i, doc in enumerate(docs_recuperados, 1):
                        st.markdown(f"**Evidência {i}** (Nota: {doc.metadata.get('review_score')} | Categoria: {doc.metadata.get('product_category_name')})")
                        st.write(f"_{doc.page_content}_")
                        st.divider()
                        
            except Exception as e:
                st.error(f"Erro ao processar a pergunta: {e}")
    else:
        st.warning("Por favor, digite uma pergunta antes de solicitar a análise.")

2026-11-01 10:03:31.522 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-01 10:03:31.523 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-01 10:03:31.523 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-01 10:03:31.524 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-01 10:03:31.524 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-01 10:03:31.524 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-01 10:03:31.526 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-01 10:03:31.528 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar